# BayesEval — Fast Facts
Auditable computation of every fact and figure cited in the paper.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
sys.path.insert(0, str(ROOT))
from analysis.scoring import (
    lifeeval_true_probability,
    medeval_true_probability,
    murphy_decomposition,
)

RESULTS = ROOT / 'results'
MODELS = [
    'anthropic_claude-haiku-4.5',
    'google_gemini-2.5-flash',
    'meta-llama_llama-4-maverick',
    'openai_gpt-5.4-mini',
]
MODEL_LABELS = ['Claude Haiku 4.5', 'Gemini 2.5 Flash', 'Llama 4 Maverick', 'GPT-5.4 Mini']

BIN_EDGES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.01]

def compute_ece(conf, outcome):
    bins = pd.cut(conf, bins=BIN_EDGES, right=False, include_lowest=True)
    g = pd.DataFrame({'conf': conf.values, 'outcome': outcome.values, 'bin': bins.values}).groupby('bin', observed=False)
    n = g.size()
    total = n.sum()
    if total == 0:
        return np.nan
    return (n / total * (g['outcome'].mean() - g['conf'].mean()).abs()).sum()

def dedup_columns(df):
    renames, drops = {}, []
    for col in df.columns:
        if col.endswith('_x'):
            base = col[:-2]
            renames[col] = base
            if f'{base}_y' in df.columns:
                drops.append(f'{base}_y')
    return df.rename(columns=renames).drop(columns=drops, errors='ignore')

def load_results(domain):
    frames = []
    for model in MODELS:
        df = dedup_columns(pd.read_csv(RESULTS / domain / f'{model}.csv'))
        df['model'] = model
        df['Confidence'] = pd.to_numeric(df['Confidence'], errors='coerce')
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

## 1. WGD Demographics

In [2]:
labels = pd.read_csv(ROOT / 'domains/WGD/Data/labels.csv')
wgd_bench = pd.read_csv(ROOT / 'domains/WGD/Data/benchmark.csv')

n_participants = len(labels)
n_photos = wgd_bench['photo'].nunique()
n_wgd_questions = len(wgd_bench)

print(f"=== WGD Demographics ===")
print(f"Participants: {n_participants}")
print(f"Unique photos: {n_photos}")
print(f"Total questions: {n_wgd_questions} ({n_photos} photos × 20 tolerances)")
print(f"\nAge:    mean={labels['Age'].mean():.1f} ± {labels['Age'].std():.1f},  median={labels['Age'].median():.1f},  range=[{labels['Age'].min():.0f}, {labels['Age'].max():.0f}]")
print(f"Weight: mean={labels['Weight'].mean():.1f} ± {labels['Weight'].std():.1f},  median={labels['Weight'].median():.1f},  range=[{labels['Weight'].min():.0f}, {labels['Weight'].max():.0f}]")
print(f"\nSex distribution:")
for sex, count in labels['sex'].value_counts().items():
    print(f"  {sex}: {count} ({count/n_participants*100:.1f}%)")
print(f"\nEthnicity distribution:")
for eth, count in labels['ethnicity'].value_counts().items():
    print(f"  {eth}: {count} ({count/n_participants*100:.1f}%)")
print(f"\nTolerance range: {wgd_bench['within_lbs'].min()}–{wgd_bench['within_lbs'].max()} lbs")

=== WGD Demographics ===
Participants: 236
Unique photos: 231
Total questions: 4620 (231 photos × 20 tolerances)

Age:    mean=24.4 ± 10.1,  median=21.0,  range=[18, 74]
Weight: mean=150.1 ± 33.1,  median=145.0,  range=[18, 380]

Sex distribution:
  Male: 127 (53.8%)
  Female: 106 (44.9%)

Ethnicity distribution:
  Asian: 101 (42.8%)
  White: 86 (36.4%)
  Indian: 23 (9.7%)
  Hispanic: 12 (5.1%)
  Black: 4 (1.7%)
  Middle Eastern: 4 (1.7%)
  Other: 3 (1.3%)

Tolerance range: 1–20 lbs


## 2. LifeEval Parameters

In [3]:
life_table_path = str(ROOT / 'domains/LifeEval/Data/PeriodLifeTable_2022_RawData.csv')
life_table = pd.read_csv(life_table_path)

le_bench = pd.read_csv(ROOT / 'domains/LifeEval/Data/benchmark.csv')

lt = life_table.dropna(subset=['Age'])
print("=== LifeEval Parameters ===")
print(f"Source: 2022 US Period Life Table (empirical rule, no parametric fit)")
print(f"\nTable ages {int(lt['Age'].min())}\u2013{int(lt['Age'].max())}; per-year death probabilities q(x):")
for age in (0, 118):
    qm = lt.loc[lt['Age'] == age, 'Death probability (MALE)'].values[0]
    qf = lt.loc[lt['Age'] == age, 'Death probability (FEMALE)'].values[0]
    print(f"  q({age}):   male = {qm:.5f},  female = {qf:.5f}")
print(f"\nLife expectancy at birth (from life table):")
print(f"  Male:   {life_table.loc[life_table['Age']==0, 'Life expectancy (MALE)'].values[0]:.2f} years")
print(f"  Female: {life_table.loc[life_table['Age']==0, 'Life expectancy (FEMALE)'].values[0]:.2f} years")
print(f"\nBenchmark: {len(le_bench)} questions (101 ages × 2 sexes × 20 radii)")
print(f"Radius range: {le_bench['radius'].min()}–{le_bench['radius'].max()} years")
print(f"\nMAS (Maximum Achievable Score):")
print(f"  Min MAS (hardest): {le_bench['MAS'].min():.6f}  (radius={le_bench.loc[le_bench['MAS'].idxmin(), 'radius']})")
print(f"  Max MAS (easiest): {le_bench['MAS'].max():.6f}  (radius={le_bench.loc[le_bench['MAS'].idxmax(), 'radius']})")
print(f"  Mean MAS by radius:")
for r in [1, 5, 10, 20]:
    sub = le_bench[le_bench['radius'] == r]
    print(f"    radius={r:2d}: mean MAS = {sub['MAS'].mean():.4f}")

=== LifeEval Parameters ===
Source: 2022 US Period Life Table (empirical rule, no parametric fit)

Table ages 0–118; per-year death probabilities q(x):
  q(0):   male = 0.00606,  female = 0.00512
  q(118):   male = 0.95797,  female = 0.95797

Life expectancy at birth (from life table):
  Male:   74.74 years
  Female: 80.18 years

Benchmark: 4040 questions (101 ages × 2 sexes × 20 radii)
Radius range: 1–20 years

MAS (Maximum Achievable Score):
  Min MAS (hardest): 0.065030  (radius=1)
  Max MAS (easiest): 1.000000  (radius=20)
  Mean MAS by radius:
    radius= 1: mean MAS = 0.1311
    radius= 5: mean MAS = 0.4832
    radius=10: mean MAS = 0.7310
    radius=20: mean MAS = 0.9343


## 3. MedEval Demographics

In [4]:
me_bench = pd.read_csv(ROOT / 'domains/MedEval/Data/benchmark_combined.csv')
me_base = me_bench[me_bench['removal_pct'] == 0].copy()

n_patients = len(me_base)
n_pathologies = me_base['true_pathology'].nunique()

# Candidates per patient
me_base['n_candidates'] = me_base['differential_json'].apply(lambda x: len(json.loads(x)))

# Mean true probability at each removal level
print("=== MedEval Demographics ===")
print(f"Total questions: {len(me_bench)} ({n_patients} patients × 4 removal levels)")
print(f"Unique patients (removal_pct=0): {n_patients}")
print(f"Unique pathologies: {n_pathologies}")
print(f"Difficulty mechanism: candidate removal + renormalization")
print(f"\nPatient age: mean={me_base['age'].mean():.1f} ± {me_base['age'].std():.1f},  range=[{me_base['age'].min()}, {me_base['age'].max()}]")
print(f"\nSex distribution:")
for sex, count in me_base['sex'].value_counts().items():
    print(f"  {sex}: {count} ({count/n_patients*100:.1f}%)")
print(f"\nCandidates per patient (at 0% removal):")
print(f"  mean={me_base['n_candidates'].mean():.1f} ± {me_base['n_candidates'].std():.1f},  range=[{me_base['n_candidates'].min()}, {me_base['n_candidates'].max()}]")

print(f"\nMean P(true_pathology) by removal level:")
for pct in [0, 10, 25, 50]:
    sub = me_bench[me_bench['removal_pct'] == pct]
    n_cands = sub['differential_json'].apply(lambda x: len(json.loads(x)))
    true_probs = [
        medeval_true_probability(row['true_pathology'], row['differential_json'])
        for _, row in sub.iterrows()
    ]
    print(f"  {pct:2d}% removed: n={len(sub)}, mean_candidates={n_cands.mean():.1f}, mean_P(true)={np.mean(true_probs):.4f}")

print(f"\nTop 10 pathologies:")
for path, count in me_base['true_pathology'].value_counts().head(10).items():
    print(f"  {path}: {count} ({count/n_patients*100:.1f}%)")

=== MedEval Demographics ===
Total questions: 768 (192 patients × 4 removal levels)
Unique patients (removal_pct=0): 192
Unique pathologies: 33
Difficulty mechanism: candidate removal + renormalization

Patient age: mean=39.6 ± 22.1,  range=[0, 105]

Sex distribution:
  M: 100 (52.1%)
  F: 92 (47.9%)

Candidates per patient (at 0% removal):
  mean=15.0 ± 4.1,  range=[10, 27]

Mean P(true_pathology) by removal level:
   0% removed: n=192, mean_candidates=15.0, mean_P(true)=0.1249
  10% removed: n=192, mean_candidates=13.0, mean_P(true)=0.1323
  25% removed: n=192, mean_candidates=10.9, mean_P(true)=0.1455
  50% removed: n=192, mean_candidates=7.3, mean_P(true)=0.1906

Top 10 pathologies:
  Stable angina: 12 (6.2%)
  Pulmonary embolism: 11 (5.7%)
  Bronchitis: 11 (5.7%)
  Acute pulmonary edema: 10 (5.2%)
  Pneumonia: 10 (5.2%)
  Anemia: 10 (5.2%)
  Anaphylaxis: 10 (5.2%)
  Epiglottitis: 9 (4.7%)
  Possible NSTEMI / STEMI: 9 (4.7%)
  Bronchospasm / acute asthma exacerbation: 9 (4.7%)


## 4. SPD Configuration

In [5]:
print("=== SPD Configuration ===\n")
for domain, spd_domain in [('WGD', 'WGD_SPD'), ('LifeEval', 'LifeEval_SPD'), ('MedEval', 'MedEval_SPD')]:
    spd_bench_name = 'benchmark_spd.csv'
    spd_path = ROOT / 'domains' / domain / 'Data' / spd_bench_name
    if not spd_path.exists():
        print(f"{domain}: SPD benchmark not found at {spd_path}")
        continue
    spd = pd.read_csv(spd_path)
    if 'photo' in spd.columns:
        # Drop photo 269.jpg: present only in the SPD build (labeling error), not in the DCE set
        spd = spd[spd['photo'] != '269.jpg']
    print(f"{domain}_SPD: {len(spd)} questions")
    if 'bin_width' in spd.columns:
        for bw in sorted(spd['bin_width'].unique()):
            sub = spd[spd['bin_width'] == bw]
            tn = sub['top_n'].iloc[0]
            print(f"  bin_width={bw:2d}, top_n={tn}, n={len(sub)}")
    if 'candidate_removal_pct' in spd.columns:
        for pct in sorted(spd['candidate_removal_pct'].unique()):
            sub = spd[spd['candidate_removal_pct'] == pct]
            print(f"  removal_pct={pct:2d}%, n={len(sub)}")
    print()

=== SPD Configuration ===



WGD_SPD: 924 questions
  bin_width= 2, top_n=10, n=231
  bin_width=10, top_n=10, n=231
  bin_width=20, top_n=5, n=231
  bin_width=40, top_n=3, n=231

LifeEval_SPD: 808 questions
  bin_width= 2, top_n=10, n=202
  bin_width=10, top_n=10, n=202


  bin_width=20, top_n=5, n=202
  bin_width=40, top_n=3, n=202

MedEval_SPD: 768 questions



## 5. RQ1 — Baseline Calibration (ECE, Brier, Overconfidence, Murphy)

In [6]:
# Load and score all baseline results
print("Loading and scoring baseline results...\n")

# WGD
wgd_df = load_results('WGD')
ans = pd.to_numeric(wgd_df['Answer'], errors='coerce')
wgd_df['outcome'] = ((ans - wgd_df['true_weight'].astype(float)).abs() <= wgd_df['within_lbs'].astype(float)).astype(float)
wgd_df.loc[ans.isna(), 'outcome'] = np.nan

# LifeEval
le_df = load_results('LifeEval')
le_ans = pd.to_numeric(le_df['Answer'], errors='coerce')
valid = le_ans.notna() & le_df['sex'].notna()
le_df['outcome'] = np.nan
le_df.loc[valid, 'outcome'] = [
    lifeeval_true_probability(a, age, sex, r)
    for a, age, sex, r in zip(le_ans[valid], le_df.loc[valid, 'min_age'], le_df.loc[valid, 'sex'], le_df.loc[valid, 'radius'])
]

# MedEval
me_df = load_results('MedEval')
valid_me = me_df['Answer'].notna() & me_df['differential_json'].notna()
me_df['outcome'] = np.nan
me_df.loc[valid_me, 'outcome'] = [
    medeval_true_probability(a, d)
    for a, d in zip(me_df.loc[valid_me, 'Answer'], me_df.loc[valid_me, 'differential_json'])
]

# Compute metrics
print(f"{'Domain':<12} {'Model':<24} {'ECE':>7} {'Brier':>7} {'Overconf':>9} {'Reliab':>7} {'Resol':>7} {'Uncert':>7}")
print("-" * 90)
for domain_name, df, outcome_col in [('WGD', wgd_df, 'outcome'), ('LifeEval', le_df, 'outcome'), ('MedEval', me_df, 'outcome')]:
    for model, label in zip(MODELS, MODEL_LABELS):
        sub = df[df['model'] == model].dropna(subset=['Confidence', outcome_col]).copy()
        conf = sub['Confidence']
        out = sub[outcome_col]
        ece = compute_ece(conf, out)
        brier = ((conf - out) ** 2).mean()
        overconf = (conf - out).mean()
        murphy = murphy_decomposition(pd.DataFrame({'Confidence': conf, 'true_probability': out}))
        print(f"{domain_name:<12} {label:<24} {ece:7.4f} {brier:7.4f} {overconf:+9.4f} {murphy['reliability']:7.4f} {murphy['resolution']:7.4f} {murphy['uncertainty']:7.4f}")

Loading and scoring baseline results...



Domain       Model                        ECE   Brier  Overconf  Reliab   Resol  Uncert
------------------------------------------------------------------------------------------
WGD          Claude Haiku 4.5          0.1228  0.2364   -0.1109  0.0163  0.0166  0.0000
WGD          Gemini 2.5 Flash          0.2170  0.2901   +0.2170  0.0514  0.0044  0.0000
WGD          Llama 4 Maverick          0.2164  0.2709   +0.2164  0.0484  0.0114  0.0000
WGD          GPT-5.4 Mini              0.0461  0.2174   +0.0160  0.0028  0.0299  0.0000
LifeEval     Claude Haiku 4.5          0.0485  0.0806   +0.0376  0.0036  0.0172  0.1399
LifeEval     Gemini 2.5 Flash          0.1210  0.0865   +0.1204  0.0157  0.0123  0.1428
LifeEval     Llama 4 Maverick          0.2086  0.1335   +0.1851  0.0562  0.0135  0.1410
LifeEval     GPT-5.4 Mini              0.0564  0.0450   -0.0327  0.0036  0.0471  0.1443
MedEval      Claude Haiku 4.5          0.6777  0.4694   +0.6777  0.4666  0.0004  0.1211
MedEval      Gemini 2.5 Flash

## 6. RQ2 — Overconfidence at Easiest vs Hardest Difficulty

In [7]:
wgd_df['overconfidence'] = wgd_df['Confidence'] - wgd_df['outcome']
le_df['overconfidence'] = le_df['Confidence'] - le_df['outcome']
me_df['overconfidence'] = me_df['Confidence'] - me_df['outcome']

print(f"{'Domain':<12} {'Model':<24} {'Hardest':>10} {'Easiest':>10}")
print("-" * 60)

for domain_name, df, diff_col, hard_val, easy_val in [
    ('WGD', wgd_df, 'within_lbs', 1, 20),
    ('LifeEval', le_df, 'radius', 1, 20),
    ('MedEval', me_df, 'removal_pct', 0, 50),
]:
    for model, label in zip(MODELS, MODEL_LABELS):
        sub = df[df['model'] == model].dropna(subset=['overconfidence'])
        hard = sub[sub[diff_col] == hard_val]['overconfidence'].mean()
        easy = sub[sub[diff_col] == easy_val]['overconfidence'].mean()
        print(f"{domain_name:<12} {label:<24} {hard:+10.4f} {easy:+10.4f}")

Domain       Model                       Hardest    Easiest
------------------------------------------------------------
WGD          Claude Haiku 4.5            +0.0633    -0.3332
WGD          Gemini 2.5 Flash            +0.3773    -0.0375
WGD          Llama 4 Maverick            +0.4396    +0.0089
WGD          GPT-5.4 Mini                +0.0248    -0.1270
LifeEval     Claude Haiku 4.5            +0.1933    -0.2488
LifeEval     Gemini 2.5 Flash            +0.3669    -0.0897
LifeEval     Llama 4 Maverick            +0.1708    -0.0858
LifeEval     GPT-5.4 Mini                +0.0785    -0.1295
MedEval      Claude Haiku 4.5            +0.6880    +0.6424
MedEval      Gemini 2.5 Flash            +0.7432    +0.6840
MedEval      Llama 4 Maverick            +0.6458    +0.6093


MedEval      GPT-5.4 Mini                +0.7949    +0.7188


## 7. RQ3 — SPD vs Baseline ECE + Bootstrap Significance

In [8]:
N_BOOT = 2000
rng = np.random.default_rng(42)

def bootstrap_ece(conf, outcome, n_boot=N_BOOT):
    n = len(conf)
    eces = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        eces[i] = compute_ece(conf.iloc[idx].reset_index(drop=True), outcome.iloc[idx].reset_index(drop=True))
    return eces

def two_sided_p(boot_delta):
    p_lower = (boot_delta <= 0).mean()
    return 2 * min(p_lower, 1 - p_lower)

# Load and score SPD results
print("Loading and scoring SPD results...\n")

wgd_spd = load_results('WGD_SPD')
# Drop photo 269.jpg: present only in the SPD build (labeling error), not in the DCE set
wgd_spd = wgd_spd[wgd_spd['photo'] != '269.jpg']
ans_s = pd.to_numeric(wgd_spd['Answer'], errors='coerce')
wgd_spd['outcome'] = ((ans_s - wgd_spd['true_weight'].astype(float)).abs() <= wgd_spd['within_lbs'].astype(float)).astype(float)
wgd_spd.loc[ans_s.isna(), 'outcome'] = np.nan

le_spd = load_results('LifeEval_SPD')
le_ans_s = pd.to_numeric(le_spd['Answer'], errors='coerce')
valid_s = le_ans_s.notna() & le_spd['sex'].notna()
le_spd['outcome'] = np.nan
le_spd.loc[valid_s, 'outcome'] = [
    lifeeval_true_probability(a, age, sex, r)
    for a, age, sex, r in zip(le_ans_s[valid_s], le_spd.loc[valid_s, 'min_age'], le_spd.loc[valid_s, 'sex'], le_spd.loc[valid_s, 'radius'])
]

me_spd = load_results('MedEval_SPD')
valid_ms = me_spd['Answer'].notna() & me_spd['differential_json'].notna()
me_spd['outcome'] = np.nan
me_spd.loc[valid_ms, 'outcome'] = [
    medeval_true_probability(a, d)
    for a, d in zip(me_spd.loc[valid_ms, 'Answer'], me_spd.loc[valid_ms, 'differential_json'])
]

# Bootstrap (two-sided test)
print(f"{'Domain':<12} {'Model':<24} {'ECE_base':>9} {'ECE_SPD':>9} {'Δ%':>8} {'p':>8} {'sig':>5} {'dir':>6}")
print("-" * 86)
for domain_name, base_df, spd_df in [('WGD', wgd_df, wgd_spd), ('LifeEval', le_df, le_spd), ('MedEval', me_df, me_spd)]:
    for model, label in zip(MODELS, MODEL_LABELS):
        sb = base_df[base_df['model'] == model].dropna(subset=['Confidence', 'outcome'])
        ss = spd_df[spd_df['model'] == model].dropna(subset=['Confidence', 'outcome'])
        ece_b = compute_ece(sb['Confidence'], sb['outcome'])
        ece_s = compute_ece(ss['Confidence'], ss['outcome'])
        if np.isnan(ece_b) or np.isnan(ece_s):
            print(f"{domain_name:<12} {label:<24} {ece_b:9.4f} {ece_s:9.4f}      N/A      N/A   N/A    N/A")
            continue
        boot_b = bootstrap_ece(sb['Confidence'], sb['outcome'])
        boot_s = bootstrap_ece(ss['Confidence'], ss['outcome'])
        boot_delta = boot_b - boot_s
        p = two_sided_p(boot_delta)
        pct = (ece_s - ece_b) / ece_b * 100
        direction = "better" if ece_s < ece_b else "worse"
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f"{domain_name:<12} {label:<24} {ece_b:9.4f} {ece_s:9.4f} {pct:+7.1f}% {p:8.4f} {sig:>5} {direction:>6}")

Loading and scoring SPD results...



Domain       Model                     ECE_base   ECE_SPD       Δ%        p   sig    dir
--------------------------------------------------------------------------------------


WGD          Claude Haiku 4.5            0.1228    0.0811   -34.0%   0.0170     * better


WGD          Gemini 2.5 Flash            0.2170    0.0717   -67.0%   0.0000   *** better


WGD          Llama 4 Maverick            0.2164    0.0736   -66.0%   0.0000   *** better


...

LifeEval     GPT-5.4 Mini                0.0564    0.0595    +5.6%   0.5510    ns  worse


MedEval      Claude Haiku 4.5            0.6777    0.3605   -46.8%   0.0000   *** better


MedEval      Gemini 2.5 Flash            0.7217    0.4196   -41.9%   0.0000   *** better


MedEval      Llama 4 Maverick            0.6374    0.3609   -43.4%   0.0000   *** better


MedEval      GPT-5.4 Mini                0.7661    0.4632   -39.5%   0.0000   *** better
